In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os

# 扩展到20只股票：科技、消费、医疗、金融各5只
tickers = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA",   # 科技
    "META", "TSLA", "AMD",  "INTC", "CRM",       # 科技2
    "JPM",  "BAC",  "GS",   "MS",   "WFC",       # 金融
    "JNJ",  "UNH",  "PFE",  "MRK",  "ABBV"       # 医疗
]

# 拉1年数据，样本量从40天 → ~250天
raw = yf.download(tickers, period="1y", interval="1d",
                  auto_adjust=True, progress=False)

close  = raw["Close"]
volume = raw["Volume"]

print("shape:", raw.shape)
print("date range:", close.index[0].date(), "→", close.index[-1].date())
print("股票数:", len(close.columns))
print("\n缺失值（每列）：")
print(close.isna().sum()[close.isna().sum() > 0])

shape: (250, 100)
date range: 2025-04-15 → 2026-04-14
股票数: 20

缺失值（每列）：
Series([], dtype: int64)


In [2]:
feature_cols = ['mom_5d', 'mom_10d', 'mom_20d', 'vol_10d', 'vol_20d',
                'rsi_dist', 'high_20d_ratio', 'vol_ratio']

def make_features(close, volume):
    feats = pd.DataFrame(index=close.index)
    feats["mom_5d"]         = close.pct_change(5)
    feats["mom_10d"]        = close.pct_change(10)
    feats["mom_20d"]        = close.pct_change(20)
    feats["vol_10d"]        = close.pct_change().rolling(10).std()
    feats["vol_20d"]        = close.pct_change().rolling(20).std()
    feats["rsi_dist"]       = (close - close.rolling(20).mean()) / close.rolling(20).std()
    feats["high_20d_ratio"] = close / close.rolling(20).max()
    feats["vol_ratio"]      = volume / volume.rolling(10).mean()
    return feats

all_feats = []
for ticker in tickers:
    feat = make_features(close[ticker], volume[ticker])
    feat["ticker"] = ticker
    feat = feat.reset_index()
    all_feats.append(feat)

df = pd.concat(all_feats, ignore_index=True).dropna()

# 加 next_ret 和 rank
next_day_ret = close.pct_change().shift(-1)
ret_long = next_day_ret.stack().reset_index()
ret_long.columns = ["Date", "ticker", "next_ret"]
df = df.merge(ret_long, on=["Date", "ticker"], how="left").dropna(subset=["next_ret"])
df["rank"] = df.groupby("Date")["next_ret"].rank(ascending=False).astype(int)

print("总行数:", len(df))
print("交易日数:", df["Date"].nunique())
print("每日股票数:", df.groupby("Date")["ticker"].count().unique())
print("rank 范围:", df["rank"].min(), "→", df["rank"].max())

总行数: 4580
交易日数: 229
每日股票数: [20]
rank 范围: 1 → 20


In [4]:
from sklearn.preprocessing import StandardScaler

# 标准化
scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

# listwise：按日期分组
groups = []
for date, group in df.groupby("Date"):
    group = group.sort_values("ticker").reset_index(drop=True)
    feats = torch.tensor(group[feature_cols].values, dtype=torch.float32)
    ranks = torch.tensor(group["rank"].values, dtype=torch.float32)
    groups.append((feats, ranks))

# pairwise：C(20,2)=190 对/天
pairs = []
for date, group in df.groupby("Date"):
    group = group.sort_values("ticker").reset_index(drop=True)
    n = len(group)
    for i in range(n):
        for j in range(i+1, n):
            fi = group.loc[i, feature_cols].values
            fj = group.loc[j, feature_cols].values
            ri, rj = group.loc[i, "rank"], group.loc[j, "rank"]
            if ri < rj:
                pairs.append((*fi, *fj, 1.0))
            elif ri > rj:
                pairs.append((*fi, *fj, 0.0))

feat_cols_i = [f"f_i_{c}" for c in feature_cols]
feat_cols_j = [f"f_j_{c}" for c in feature_cols]
pair_df = pd.DataFrame(pairs, columns=feat_cols_i + feat_cols_j + ["label"])

# 切分：前183天(80%)训练，后46天(20%)验证
train_groups = groups[:183]
val_groups   = groups[183:]

pair_df["split"] = "val"
train_dates = sorted(df["Date"].unique())[:183]
# 重新构造 pair_df 带 split 标记
train_pair = pair_df.iloc[:183*190]
val_pair   = pair_df.iloc[183*190:]

print(f"listwise  — train: {len(train_groups)} 天, val: {len(val_groups)} 天")
print(f"pairwise  — train: {len(train_pair)} 对, val: {len(val_pair)} 对")
print(f"pairwise train label 分布: {train_pair['label'].value_counts().to_dict()}")

listwise  — train: 183 天, val: 46 天
pairwise  — train: 34770 对, val: 8740 对
pairwise train label 分布: {1.0: 17400, 0.0: 17370}


In [5]:
from torch.utils.data import DataLoader, TensorDataset

# --- RankNet ---
class RankNet(nn.Module):
    def __init__(self, input_dim=8):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, fi, fj):
        return torch.sigmoid(self.scorer(fi) - self.scorer(fj)).squeeze(1)
    def score(self, f):
        return self.scorer(f).squeeze(1)

# --- ListNet ---
class ListNet(nn.Module):
    def __init__(self, input_dim=8):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.scorer(x).squeeze(1)

def listnet_loss(scores, ranks):
    p_true = torch.softmax(-ranks, dim=0)
    p_pred = torch.softmax(scores, dim=0)
    return -torch.sum(p_true * torch.log(p_pred + 1e-9))

def ndcg_at_k(pred_scores, true_ranks, k=5):
    order = torch.argsort(pred_scores, descending=True)
    true_relevance = (21 - true_ranks).float()  # rank1→20, rank20→1
    dcg  = sum((true_relevance[order[i]] / np.log2(i+2)).item() for i in range(k))
    ideal = torch.argsort(true_relevance, descending=True)
    idcg = sum((true_relevance[ideal[i]] / np.log2(i+2)).item() for i in range(k))
    return dcg / idcg if idcg > 0 else 0.0

# pairwise tensors
def make_tensors(df):
    fi    = torch.tensor(df[feat_cols_i].values, dtype=torch.float32)
    fj    = torch.tensor(df[feat_cols_j].values, dtype=torch.float32)
    label = torch.tensor(df["label"].values,     dtype=torch.float32)
    return fi, fj, label

fi_tr, fj_tr, y_tr = make_tensors(train_pair)
fi_va, fj_va, y_va = make_tensors(val_pair)
train_loader = DataLoader(TensorDataset(fi_tr, fj_tr, y_tr), batch_size=256, shuffle=True)

# 训练 RankNet
model_rn = RankNet(input_dim=8)
opt_rn   = torch.optim.Adam(model_rn.parameters(), lr=1e-3)
bce      = nn.BCELoss()

for epoch in range(50):
    model_rn.train()
    for fi_b, fj_b, y_b in train_loader:
        opt_rn.zero_grad()
        bce(model_rn(fi_b, fj_b), y_b).backward()
        opt_rn.step()

model_rn.eval()
with torch.no_grad():
    val_acc = ((model_rn(fi_va, fj_va) > 0.5) == y_va.bool()).float().mean().item()
    rn_ndcg = np.mean([ndcg_at_k(model_rn.score(f), r, k=5) for f, r in val_groups])
print(f"RankNet  — val_acc: {val_acc:.4f} | NDCG@5: {rn_ndcg:.4f}")

# 训练 ListNet
model_ln = ListNet(input_dim=8)
opt_ln   = torch.optim.Adam(model_ln.parameters(), lr=1e-3)
best_val, best_state, no_improve = float("inf"), None, 0

for epoch in range(200):
    model_ln.train()
    opt_ln.zero_grad()
    loss = sum(listnet_loss(model_ln(f), r) for f, r in train_groups)
    loss.backward()
    opt_ln.step()
    
    model_ln.eval()
    with torch.no_grad():
        val_loss = sum(listnet_loss(model_ln(f), r).item() for f, r in val_groups) / len(val_groups)
    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.clone() for k, v in model_ln.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
    if no_improve >= 20:
        print(f"Early stopping @ epoch {epoch+1}, best val_loss: {best_val:.4f}")
        break

model_ln.load_state_dict(best_state)
model_ln.eval()
with torch.no_grad():
    ln_ndcg = np.mean([ndcg_at_k(model_ln(f), r, k=5) for f, r in val_groups])
print(f"ListNet  — NDCG@5: {ln_ndcg:.4f}")
print(f"\n随机基准 NDCG@5 ≈ {1/5:.4f}")

RankNet  — val_acc: 0.4904 | NDCG@5: 0.5403
Early stopping @ epoch 69, best val_loss: 2.8222
ListNet  — NDCG@5: 0.6025

随机基准 NDCG@5 ≈ 0.2000


In [6]:
save_dir = os.path.expanduser("~/ml-projects/stock-ranker/data")

torch.save({"model_state": model_rn.state_dict(),
            "hparams": {"input_dim": 8, "feature_cols": feature_cols}},
           f"{save_dir}/ranknet_v2.pth")

torch.save({"model_state": model_ln.state_dict(),
            "hparams": {"input_dim": 8, "feature_cols": feature_cols}},
           f"{save_dir}/listnet_v2.pth")

import pickle
with open(f"{save_dir}/scaler_v2.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("已保存：ranknet_v2.pth / listnet_v2.pth / scaler_v2.pkl")

已保存：ranknet_v2.pth / listnet_v2.pth / scaler_v2.pkl
